
**Building a RAG applicaiton in DBx**

![image_1780951888176.png](./image_1780951888176.png "image_1780951888176.png")

In [0]:
# Install libraries and packages
%pip install databricks-vectorsearch openapi mlflow

In [0]:
dbutils.library.restartPython() # Restart python kernel to catch the install libraries

In [0]:
import mlflow
from mlflow import pyfunc
from openai import OpenAI

class RAGMODEL(pyfunc.PythonModel):
    def __init__(self, vector_index):
        self.vector_index = vector_index

    def retrieve(self, query):
        results = self.vector_index.similar_search(
            query_text= query,
            columns = ["id", "content_path", "chunk"],
            num_results = 10
        )
    def chatCompletionsAPI(self, user_query, supporting_knowledge): 
        openai_client = OpenAI(
            api_key="UR DATABRICKS ACCESS TOKEN",
            base_url="https://adb-1884344040313130.10.azuredatabricks.net/serving-endpoints"
        )

        completion = openai_client.chat.completions.create(
            model = "databricks-claude-haiku-4-6", # Choose your preferred model
            messages=[
                {
                    "role": "system",
                    "content": [
                        {
                            "type": "text",
                            "text":"You are a helpful assistant. You will be provided with a user query and a set of supporting knowledge. Your task is to generate a clear, accurate, and concise response based strictly on the given information. Do not fabricate details, and indicate if the answer cannot be fully derived from the provided context." 
                        }                        
                    ]
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"User query: {user_query}\n\nSupporting knowledge: {supporting_knowledge}"
                        }
                    ]
                }
            ]
        )

        return completion.choices[0].message.content

    def predict(self, context, model_input):
        query = model_input["query"][0]
        text_data = self.retrieve(query)
        return self.chatCompletionsAPI(query, text_data)


  
  

In [0]:
# Fetch the vector index using Mosaic AI vector client
from databricks.vector_search.client import VectorSearchClient

vector_client = VectorSearchClient()
vector_index = vector_client.get_index(index_name="UR_CATALOG_NAME.rag.rag_vector_index")  # make sure this matches your vector index in Unity Catalog


In [0]:
# Save the the RAG model (AI agent)
import pandas as pd
from mlflow.models import infer_signature

# Instantiate the RAG model
rag_model = RAGMODEL(vector_index)

#Sample input
input_example = pd.DataFrame({
    "user_query": "Hi How are you?"
})

# Sample output
output_example = pd.DataFrame({
    "response": "Hello! I'm doing well, how about you?"
})

#Infer full signature (input + output)
signature = infer_signature(input_example, output_example)

#Save the model
mlflow.pyfunc.save_model(
    path="rag_model",
    python_model=rag_model,
    signature=signature,
    input_example=input_example,
    metadata={"model": "rag_model"}
)
 

In [0]:
# Loading our saved model 
loaded_model = mlflow.pyfunc.load_model("rag_model")


In [0]:
# Testing our model
model_input = pd.DataFrame([{
    "user_query": "Tell me something about the hotels offered by Margies Travels in Dubai"
}])

model_response = loaded_model.predict(pd.DataFrame({"user_query": ["Hi How are you?"]}))
print(model_response)

In [0]:
# Logging our Saved Model as an Artifact
import mlflow
run_id = None

#Log the model as an artifact
with mlflow.start_run() as run:
    mlflow.log_artifact("rag_model")
    run_id = run.info.run_id
# Registering our model in MLFLow
model_name = "rag_model"
model = mlflow.register_model(f"runs:/{run_id}/rag_model", model_name)

In [0]:
%skip
# Transitioning our model to Production
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.transition_model_version_stage(
    name=model_name,
    version=model.version,
    stage="Production",
    archive_existing_versions=True,
    comments="This is the best model for our RAG use case."
)

In [0]:
# Inferencing the deployed Real-time Endpoint: Use the following payload sample to test your endpoint
{
  "dataframe_split": {
    "columns": [
      "user_query"
    ],
    "data": [
      [
        "Tell me something about the hotels offered by Margies Travels in Dubai"
      ]
    ]
  }
}
